In [4]:
import pynetlogo
import pandas as pd
import itertools
from pathlib import Path

# 1. Explicitly point to the jvm.dll file inside NetLogo's directory
jvm_path = "C:/Program Files/NetLogo 6.4.0/runtime/bin/server/jvm.dll"

netlogo = pynetlogo.NetLogoLink(
    gui=False,
    netlogo_home="C:/Program Files/NetLogo 6.4.0",
    jvm_path=jvm_path 
)

model_path = "C:/Users/15177459/Desktop/netlogo/models/survey_derived.nlogo"

netlogo.load_model(model_path)
netlogo.command("setup")

print("People:", netlogo.report("count people"))
print("Places:", netlogo.report("count places"))
print("Ticks:", netlogo.report("ticks"))

People: 300.0
Places: 60.0
Ticks: 0.0


In [5]:
parameter_grid = {
    "number-of-people": [100, 200, 300],
    "baseline-stop-probability": [18, 30, 40],
    "third-place-search-radius": [1, 2, 5],
    "route-third-place-sample-size": [50],
    "rigid-dwell-time": [30],
    "medium-dwell-time": [45],
    "flexible-dwell-time": [60],
    "encounter-odds-increment": [0.05, 0.10, 0.20],
    "encounter-weight": [1, 5, 8],
    "social-feedback?": [True],
}

reporters = [
    "average-route-third-place-options",
    "people-with-route-third-place-options",
    "visits-per-person",
    "average-stop-probability",
    "average-social-encounters",
    "average-social-odds-ratio",
    "total-third-place-visits",
    "total-co-presence",
    "co-presence-per-visit",
    "places-with-co-presence",
    "share-places-visited",
    "rigid-visits-per-person",
    "medium-visits-per-person",
    "flexible-visits-per-person"
]

results = []

keys = list(parameter_grid.keys())
values = list(parameter_grid.values())

run_id = 0

for combination in itertools.product(*values):
    params = dict(zip(keys, combination))
    
    for seed in range(1, 6):  # start with 5 repetitions
        run_id += 1
        
        netlogo.command(f"random-seed {seed}")
        
        for parameter, value in params.items():
            if isinstance(value, bool):
                value = "true" if value else "false"
            netlogo.command(f"set {parameter} {value}")
        
        # Link rigid/flexible dwell time to medium dwell time
        medium_dwell = params["medium-dwell-time"]
        netlogo.command(f"set rigid-dwell-time {max(5, medium_dwell * 0.5)}")
        netlogo.command(f"set flexible-dwell-time {medium_dwell * 1.5}")
        
        netlogo.command("setup")
        netlogo.command("repeat 1000 [ go ]")
        
        row = {
            "run_id": run_id,
            "seed": seed,
            **params,
        }
        
        for reporter in reporters:
            row[reporter] = netlogo.report(reporter)
        
        results.append(row)

df = pd.DataFrame(results)
df.head(20)

,run_id,seed,number-of-people,baseline-stop-probability,third-place-search-radius,route-third-place-sample-size,rigid-dwell-time,medium-dwell-time,flexible-dwell-time,encounter-odds-increment,...,average-social-encounters,average-social-odds-ratio,total-third-place-visits,total-co-presence,co-presence-per-visit,places-with-co-presence,share-places-visited,rigid-visits-per-person,medium-visits-per-person,flexible-visits-per-person
0,1,1,100,18,1,50,30,45,60,0.05,...,0.12,1.006,91.0,322.0,3.538462,4.0,0.750000,0.862069,0.750000,1.114286
1,2,2,100,18,1,50,30,45,60,0.05,...,0.22,1.011,135.0,570.0,4.222222,8.0,0.866667,1.057143,1.457143,1.566667
2,3,3,100,18,1,50,30,45,60,0.05,...,0.44,1.022,141.0,1109.0,7.865248,14.0,0.833333,1.085714,1.529412,1.645161
3,4,4,100,18,1,50,30,45,60,0.05,...,0.16,1.008,93.0,390.0,4.193548,6.0,0.783333,0.774194,0.787879,1.194444
4,5,5,100,18,1,50,30,45,60,0.05,...,0.36,1.018,132.0,916.0,6.939394,12.0,0.900000,1.268293,1.272727,1.461538
5,6,1,100,18,1,50,30,45,60,0.05,...,0.80,1.040,103.0,407.0,3.951456,6.0,0.766667,0.827586,0.944444,1.285714
6,7,2,100,18,1,50,30,45,60,0.05,...,0.90,1.045,138.0,301.0,2.181159,6.0,0.866667,1.057143,1.600000,1.500000
7,8,3,100,18,1,50,30,45,60,0.05,...,1.70,1.085,138.0,786.0,5.695652,11.0,0.816667,1.085714,1.529412,1.548387
8,9,4,100,18,1,50,30,45,60,0.05,...,0.80,1.040,93.0,390.0,4.193548,6.0,0.783333,0.774194,0.787879,1.194444
9,10,5,100,18,1,50,30,45,60,0.05,...,2.00,1.100,140.0,864.0,6.171429,14.0,0.816667,1.243902,1.484848,1.538462


In [7]:
output_path = "C:/Users/15177459/Desktop/netlogo/models/outputs/sweep_results_test.csv"

df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(df.shape)

Saved to: C:/Users/15177459/Desktop/netlogo/models/outputs/sweep_results_test.csv
(1215, 26)


In [8]:
summary = (
    df.groupby([
        "number-of-people",
        "baseline-stop-probability",
        "medium-dwell-time",
        "encounter-odds-increment",
        "encounter-weight"
    ])
    [[
        "visits-per-person",
        "average-stop-probability",
        "average-social-encounters",
        "average-social-odds-ratio",
        "total-co-presence",
        "co-presence-per-visit"
    ]]
    .mean()
    .reset_index()
)

summary.sort_values(
    by=["average-social-odds-ratio", "average-social-encounters"],
    ascending=False
).head(20)

,number-of-people,baseline-stop-probability,medium-dwell-time,encounter-odds-increment,encounter-weight,visits-per-person,average-stop-probability,average-social-encounters,average-social-odds-ratio,total-co-presence,co-presence-per-visit
80,300,40,45,0.20,8,2.653778,50.253987,28.042667,2.373778,16900.600000,21.074258
79,300,40,45,0.20,5,2.623556,49.425041,17.675556,2.292356,16770.400000,21.136861
77,300,40,45,0.10,8,2.584000,48.332612,26.524444,2.200400,15933.133333,20.386989
53,200,40,45,0.20,8,2.449667,46.986264,15.013333,2.139467,6794.466667,13.674800
71,300,30,45,0.20,8,2.100889,36.928280,18.314667,2.118133,11585.133333,18.193956
52,200,40,45,0.20,5,2.432667,45.946642,9.793333,2.035267,7037.266667,14.240485
76,300,40,45,0.10,5,2.534000,46.316241,15.966667,2.026867,15357.133333,20.034542
70,300,30,45,0.20,5,2.051333,35.560313,10.775556,1.989822,11187.466667,18.019202
68,300,30,45,0.10,8,2.026000,34.971936,17.521778,1.932311,11125.466667,17.993414
74,300,40,45,0.05,8,2.495333,44.967663,24.750222,1.909289,15268.133333,20.225904


In [9]:
weight_effect = (
    df.groupby("encounter-weight")
    [[
        "visits-per-person",
        "average-stop-probability",
        "average-social-encounters",
        "average-social-odds-ratio",
        "total-co-presence",
        "co-presence-per-visit"
    ]]
    .mean()
    .reset_index()
)

weight_effect

,encounter-weight,visits-per-person,average-stop-probability,average-social-encounters,average-social-odds-ratio,total-co-presence,co-presence-per-visit
0,1,1.703593,24.466240,1.058132,1.123022,4540.158025,10.739573
1,5,1.815934,29.028246,6.100617,1.481468,5166.938272,11.273739
2,8,1.857366,30.553699,10.148675,1.616742,5365.661728,11.456707


In [10]:
feedback_strength = (
    df.groupby([
        "encounter-weight",
        "encounter-odds-increment"
    ])
    [[
        "visits-per-person",
        "average-stop-probability",
        "average-social-encounters",
        "average-social-odds-ratio",
        "total-co-presence",
        "co-presence-per-visit"
    ]]
    .mean()
    .reset_index()
)

feedback_strength.sort_values(
    by=["encounter-weight", "encounter-odds-increment"]
)

,encounter-weight,encounter-odds-increment,visits-per-person,average-stop-probability,average-social-encounters,average-social-odds-ratio,total-co-presence,co-presence-per-visit
0,1,0.05,1.676901,23.371726,1.027975,1.051399,4402.533333,10.620400
1,1,0.10,1.698728,24.225122,1.046321,1.104311,4493.570370,10.696084
2,1,0.20,1.735148,25.801872,1.100099,1.213356,4724.370370,10.902236
3,5,0.05,1.757840,26.492362,5.613457,1.265125,4823.414815,10.931252
4,5,0.10,1.809728,29.063824,6.074198,1.477798,5118.303704,11.264021
5,5,0.20,1.880235,31.528551,6.614198,1.701481,5559.096296,11.625945
6,8,0.05,1.793506,28.204835,9.446519,1.402356,5059.777778,11.274826
7,8,0.10,1.858877,30.753436,10.208198,1.629015,5372.162963,11.394950
8,8,0.20,1.919716,32.702825,10.791309,1.818857,5665.044444,11.700346
